<a href="https://colab.research.google.com/github/carlosmenendez10/Conversational-AI-Agent/blob/main/Conversational_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
import random
from datetime import datetime, timedelta

# Conexion a SQLite
conn = sqlite3.connect('banca_sintetica_robusta.db')
cursor = conn.cursor()

# Estructura de las tablas
cursor.executescript('''
    DROP TABLE IF EXISTS movimientos;
    DROP TABLE IF EXISTS creditos;
    DROP TABLE IF EXISTS cuentas;
    DROP TABLE IF EXISTS clientes;

    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        segmento TEXT
    );

    CREATE TABLE cuentas (
        id_cuenta TEXT PRIMARY KEY,
        id_cliente INTEGER,
        tipo_producto TEXT,
        saldo_actual REAL,
        FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
    );

    CREATE TABLE creditos (
        id_credito TEXT PRIMARY KEY,
        id_cliente INTEGER,
        tipo_credito TEXT,
        etapa_riesgo INTEGER,
        saldo_insoluto REAL,
        dias_vencido INTEGER,
        FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
    );

    CREATE TABLE movimientos (
        id_movimiento INTEGER PRIMARY KEY AUTOINCREMENT,
        id_cuenta TEXT,
        fecha TEXT,
        concepto TEXT,
        categoria TEXT,
        monto REAL
    );
''')

# Datos Dummy
cursor.executemany("INSERT INTO clientes VALUES (?, ?, ?)", [
    (1, 'Ernesto Fernández', 'Select'),
    (2, 'Daniela Pérez', 'Clásico')
])

cursor.executemany("INSERT INTO cuentas VALUES (?, ?, ?, ?)", [
    ('TDC-1234', 1, 'Tarjeta de Crédito LikeU', 15000.00),
    ('DEB-5678', 1, 'Cuenta de Débito', 24500.50),
    ('TDC-9012', 2, 'Tarjeta de Crédito Clásica', 5000.00)
])

cursor.executemany("INSERT INTO creditos VALUES (?, ?, ?, ?, ?, ?)", [
    ('CRED-001', 1, 'Consumo', 1, 80000.00, 0),
    ('CRED-002', 2, 'Vivienda', 2, 1400000.00, 45)
])

# Generar algunos movimientos de prueba
movimientos_data = [
    ('TDC-1234', '2026-09-28', 'Sushiroll', 'Restaurantes', 450.00),
    ('TDC-1234', '2026-09-29', 'Estacionamiento', 'Transporte', 45.00),
    ('TDC-1234', '2026-09-30', 'Restaurante Somma', 'Restaurantes', 1200.00),
    ('DEB-5678', '2026-09-30', 'Nomina', 'Ingresos', 15000.00)
]
cursor.executemany("INSERT INTO movimientos (id_cuenta, fecha, concepto, categoria, monto) VALUES (?, ?, ?, ?, ?)", movimientos_data)
conn.commit()

print("Base de datos creada")

Base de datos creada


In [9]:
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Configuración API
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Lista de modelos disponibles
print("Modelos disponibles:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Modelos disponibles:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flas

In [10]:
import google.generativeai as genai
from google.colab import userdata

# Sacamos la API Key de forma segura
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Probamos el modelo
model = genai.GenerativeModel('gemini-3.5-flash')
respuesta = model.generate_content("Hola, dime en una oración qué es un Agente de Inteligencia Artificial.")
print(respuesta.text)

Un Agente de Inteligencia Artificial es un sistema autónomo capaz de percibir su entorno, procesar información y realizar acciones para alcanzar un objetivo específico.


In [11]:
import google.generativeai as genai

print("Mis modelos de Embeddings disponibles son:")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

Mis modelos de Embeddings disponibles son:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [12]:
import requests
from google.colab import userdata
import numpy as np
import google.generativeai as genai

# Variables de configuración GitHub
usuario_github = "carlosmenendez10"
repositorio = "Conversational-AI-Agent"
rama = "main"
archivo = "politicas_banco.txt"

# Llave de GitHub (secrerto)
github_token = userdata.get('GITHUB_TOKEN')

# URL del archivo
url_github = f"https://raw.githubusercontent.com/{usuario_github}/{repositorio}/{rama}/{archivo}"

# Conexion d github
headers = {'Authorization': f'token {github_token}'}

print("Conectando con GitHub...")
respuesta_github = requests.get(url_github, headers=headers)

if respuesta_github.status_code == 200:
    print("Archivo leído desde el repositorio")
    texto_leido = respuesta_github.text
else:
    print(f"Error al conectar con GitHub. Código: {respuesta_github.status_code}")
    print("Verifica tu usuario, el nombre del repo o tu token.")
    texto_leido = ""

# Motor RAG
if texto_leido:
    fragmentos = [frag.strip() for frag in texto_leido.split('\n\n') if frag.strip() != '']

    def obtener_embedding(texto, tipo_tarea="retrieval_document"):
        resultado = genai.embed_content(
            model="models/gemini-embedding-2",
            content=texto,
            task_type=tipo_tarea
        )
        return resultado['embedding']

    print("Generando embeddings desde la nube...")
    embeddings_base = [obtener_embedding(frag) for frag in fragmentos]
    print("Motor RAG listo y conectado a GitHub")

    def buscar_informacion(pregunta_usuario):
        vector_pregunta = obtener_embedding(pregunta_usuario, tipo_tarea="retrieval_query")
        similitudes = np.dot(embeddings_base, vector_pregunta)
        indice_mejor_resultado = np.argmax(similitudes)
        return fragmentos[indice_mejor_resultado]

    # PRUEBA DEL MOTOR RAG NUBE
    pregunta = "Necesito obligatoriamente un seguro de vida para mi hipoteca?"
    print(f"\nPregunta: {pregunta}")
    print(f"Contexto recuperado:\n{buscar_informacion(pregunta)}")

Conectando con GitHub...
Archivo leído desde el repositorio
Generando embeddings desde la nube...
Motor RAG listo y conectado a GitHub

Pregunta: Necesito obligatoriamente un seguro de vida para mi hipoteca?
Contexto recuperado:
SECCIÓN 2: CRÉDITOS DE VIVIENDA (HIPOTECARIOS)
Condiciones de Originación: Todos los créditos de vivienda requieren un enganche mínimo del 10% del valor del inmueble. El plazo máximo de financiamiento es de 20 años. Se requiere la contratación de un seguro de vida y daños vinculados al crédito.


In [14]:
########## API INTERACTIONS V2 #####################
import sqlite3
import requests
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata

# 1. Inicializamos el NUEVO cliente (ahora sirve para Chat y Embeddings)
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# 2. Descargamos el TXT desde GitHub
usuario_github = "carlosmenendez10" # <-- REEMPLAZA
repositorio = "Conversational-AI-Agent"
rama = "main"
archivo = "politicas_banco.txt"
github_token = userdata.get('GITHUB_TOKEN')

url_github = f"https://raw.githubusercontent.com/{usuario_github}/{repositorio}/{rama}/{archivo}"
headers = {'Authorization': f'token {github_token}'}
respuesta_github = requests.get(url_github, headers=headers)
texto_leido = respuesta_github.text

# Hacemos el Chunking
fragmentos = [frag.strip() for frag in texto_leido.split('\n\n') if frag.strip() != '']

# 3. Actualizamos la función de Embeddings a la sintaxis del nuevo SDK
def obtener_embedding_nuevo(texto):
    respuesta = client.models.embed_content(
        model='gemini-embedding-2', # Usamos el modelo estándar de embeddings
        contents=texto
    )
    # El nuevo SDK devuelve los valores en esta ruta:
    return respuesta.embeddings[0].values

print("Generando embeddings con el nuevo SDK...")
embeddings_base = [obtener_embedding_nuevo(frag) for frag in fragmentos]

# 4. Envolvemos las Herramientas (Tools) con manejo de errores interno
def consultar_base_datos(consulta_sql: str) -> str:
    """
    Ejecuta una consulta SQL en la base de datos del Banco y devuelve el resultado.
    Úsala EXCLUSIVAMENTE cuando el usuario pregunte por sus transacciones, saldos, productos o movimientos.

    Tablas disponibles:
    - clientes (id_cliente, nombre, segmento)
    - cuentas (id_cuenta, id_cliente, tipo_producto, saldo_actual)
    - creditos (id_credito, id_cliente, tipo_credito, etapa_riesgo, saldo_insoluto, dias_vencido)
    - movimientos (id_movimiento, id_cuenta, fecha, concepto, categoria, monto)

    REGLAS DE ORO (¡SÍGUELAS ESTRICTAMENTE!):
    1. "Productos" significa tanto lo que hay en la tabla 'cuentas' como en la tabla 'creditos'.
    2. Si te preguntan por los productos de un cliente, debes consultar AMBAS tablas (con UNION o en consultas separadas).
    3. NUNCA inventes productos, créditos, cuentas ni saldos. Solo menciona EXACTAMENTE lo que el resultado SQL te devuelva. Si el SQL te devuelve 2 productos, responde que tiene 2 productos. CERO alucinaciones.
    """
    print(f"\n[DEBUG SQL] La IA generó esta consulta: {consulta_sql}")
    try:
        conn = sqlite3.connect('banca_sintetica_robusta.db')
        cursor = conn.cursor()
        cursor.execute(consulta_sql)
        resultados = cursor.fetchall()
        conn.close()
        print(f"[DEBUG SQL] Resultado de la base de datos: {resultados}")
        return str(resultados)
    except Exception as e:
        print(f"\n[DEBUG SQL] ¡ERROR EN CONSULTA!: {e}")
        return f"Error SQL: {e}"

def consultar_politicas(pregunta_usuario: str) -> str:
    """
    Busca en las políticas internas del Banco y devuelve el fragmento más relevante.
    Úsala EXCLUSIVAMENTE cuando el usuario pregunte por reglas, anualidades, penalizaciones o beneficios.
    """
    print(f"\n[DEBUG RAG] La IA decidió buscar esta frase: '{pregunta_usuario}'")

    try:
        # 1. Convertimos la pregunta
        vector_pregunta = obtener_embedding_nuevo(pregunta_usuario)

        # 2. Cálculo de Similitud del Coseno (Matemática más precisa)
        A = np.array(embeddings_base)
        B = np.array(vector_pregunta)

        normas_A = np.linalg.norm(A, axis=1)
        norma_B = np.linalg.norm(B)
        similitudes = np.dot(A, B) / (normas_A * norma_B)

        # 3. Obtenemos el mejor resultado
        indice_mejor = np.argmax(similitudes)
        fragmento_elegido = fragmentos[indice_mejor]

        print(f"[DEBUG RAG] Fragmento recuperado exitosamente: {fragmento_elegido[:75]}...")
        return fragmento_elegido

    except Exception as e:
        # Si algo explota, lo imprimimos en rojo para que nosotros lo veamos
        print(f"\n[DEBUG RAG] ¡ERROR OCULTO DESCUBIERTO!: {e}")
        return f"Dile al cliente que hubo un error técnico interno: {e}"

# 5. Creamos el Agente con la API de Interactions
chat = client.chats.create(
    model="gemini-3.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction="Eres un asesor financiero experto del Banco. Sé amable, claro y directo.",
        tools=[consultar_base_datos, consultar_politicas],
        temperature=0.2, # Temperatura baja para que sea más preciso y menos creativo
    )
)

# 6. Prueba Final
mensaje_cliente = """
Hola. Quiero saber algunas cosas:
Mi número de cliente (ID) es 2.
1. Cuántos productos tengo contratados? Y qué tipos de productos son (Busca productos que tengan cliente con ID 1)
2. Dime cuánto dinero exacto gasté en la categoría 'Restaurantes' y dame el desglose.
"""

print("El Agente está pensando y gestionando herramientas en la nube...\n")
respuesta_agente = chat.send_message(mensaje_cliente)

print("--- RESPUESTA FINAL DEL AGENTE ---")
print(respuesta_agente.text)

Generando embeddings con el nuevo SDK...
El Agente está pensando y gestionando herramientas en la nube...


[DEBUG SQL] La IA generó esta consulta: SELECT 'cuenta' AS tipo_producto, tipo_producto AS detalle FROM cuentas WHERE id_cliente = 1 UNION ALL SELECT 'credito' AS tipo_producto, tipo_credito AS detalle FROM creditos WHERE id_cliente = 1;
[DEBUG SQL] Resultado de la base de datos: [('cuenta', 'Tarjeta de Crédito LikeU'), ('cuenta', 'Cuenta de Débito'), ('credito', 'Consumo')]

[DEBUG SQL] La IA generó esta consulta: SELECT m.fecha, m.concepto, m.monto FROM movimientos m JOIN cuentas c ON m.id_cuenta = c.id_cuenta WHERE c.id_cliente = 2 AND m.categoria = 'Restaurantes';
[DEBUG SQL] Resultado de la base de datos: []
--- RESPUESTA FINAL DEL AGENTE ---
Hola. Con mucho gusto te ayudo a revisar la información que solicitas:

1. **Productos contratados:**
   Tomando en cuenta las instrucciones específicas que me diste de revisar los productos asociados al cliente con **ID 1**, actualment